In [ ]:
!pip3 install -U ucimlrepo 
%pip install scikit-learn


  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl (8.4 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl (37.4 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ ---------

## 2. Helper functions for repeated experiments

These functions only automate **evaluation and plotting**. The SVM model itself is still scikit-learn's `SVC`.

In [ ]:
#standardized data set using scaler and pipeline for SVM model, 3 variables:  C, degree, gamma
def svm_pipeline(kernel, C=1.0, degree=3, gamma="scale"):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel=kernel, C=C, degree=degree, gamma=gamma))
    ])

#separate the data into training and testing sets 
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train) # run optimization to find the best hyperplane
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    support_count = int(model.named_steps["svm"].n_support_.sum())
    return {
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Support Vectors": support_count
    }

# decision boundary plot for the SVM model from sklearn.inspection.DecisionBoundaryDisplay
def plot_boundary(model, X, y, title):
    plt.figure(figsize=(7, 5))
    DecisionBoundaryDisplay.from_estimator(
        model,
        X,
        response_method="predict",
        alpha=0.25
    )
    plt.scatter(X[:, 0], X[:, 1], c=y, s=28)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.show()

## 1. Load and inspect the dataset

**Dataset:** Breast Cancer Wisconsin dataset from `scikit-learn`.


In [9]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets

# put features and target side by side in one DataFrame
df = pd.concat([X, y], axis=1)

print(df.shape)   # (4424, 37): 36 features + 1 target
print(X.shape)    # features only
print(y.shape)    # target only
df.head()



(4424, 37)
(4424, 36)
(4424, 1)


,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 2. Create one train/test split for all models

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

NameError: name 'RANDOM_STATE' is not defined

# Part A — Approximately Linear Data

We first generate a two-class dataset that is approximately linearly separable. To make preprocessing visible, the second feature is deliberately placed on a much larger numerical scale.

In [16]:
from sklearn.datasets import make_blobs, make_moons, make_circles

RANDOM_STATE = 42

X_linear, y_linear = make_blobs(
    n_samples=300,
    centers=[(-2, -2), (2, 2)],
    cluster_std=1.4,
    random_state=RANDOM_STATE
)
# Display the first 10 target values

# Deliberately create very different feature scales.
X_linear[:, 1] = X_linear[:, 1] * 80

linear_df = pd.DataFrame(X_linear, columns=["Feature_1", "Feature_2"])
linear_df["Target"] = y_linear

display(linear_df.head())
print("Shape:", linear_df.shape)
print()
print("Missing values:")
print(linear_df.isna().sum())
print()
print("Duplicates:", linear_df.duplicated().sum())
print()
print("Class counts:")
print(linear_df["Target"].value_counts().sort_index())
display(linear_df[["Feature_1", "Feature_2"]].describe())

,Feature_1,Feature_2,Target
0,-3.667825,-86.465996,0
1,-2.762136,-147.576670,0
2,2.168414,217.617149,1
3,-3.374112,-108.244411,0
4,-1.918508,-288.012673,0


Shape: (300, 3)

Missing values:
Feature_1    0
Feature_2    0
Target       0
dtype: int64

Duplicates: 0

Class counts:
Target
0    150
1    150
Name: count, dtype: int64


,Feature_1,Feature_2
count,300.000000,300.000000
mean,-0.015098,-1.820474
std,2.454342,189.729393
min,-6.537774,-382.607718
25%,-1.984558,-145.911959
50%,-0.053054,8.003990
75%,2.101676,143.660554
max,6.310433,414.317600


## 3. Train/test split and scaling

The split is created **before** the scaler learns any parameters. Using a `Pipeline` ensures that `StandardScaler` is fitted only on training data and then applied consistently to test data.

In [18]:
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_linear,
    y_linear,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_linear
)

linear_svm = svm_pipeline(kernel="linear", C=1.0)
linear_svm.fit(X_train_lin, y_train_lin)

print("Training accuracy:", accuracy_score(y_train_lin, linear_svm.predict(X_train_lin)))
print("Test accuracy:", accuracy_score(y_test_lin, linear_svm.predict(X_test_lin)))
print("Support vectors by class:", linear_svm.named_steps["svm"].n_support_)

NameError: name 'svm_pipeline' is not defined